# V2 — استخراج فریم برای آموزش چندموقعیتی

این notebook فقط ۱٬۴۴۰ sequence جدیدِ train را decode و cache می‌کند. ۱۲۰ sequence ثابتِ validation از cache سالم V2-W2 دوباره استفاده می‌شوند؛ بنابراین split و دادهٔ اعتبارسنجی تغییر نمی‌کنند.

In [1]:
from __future__ import annotations

from pathlib import Path
import json

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

DATA_ROOT = Path(r'P:\\NexarCollisionData')
MULTIPOS_MANIFEST_PATH = DATA_ROOT / 'sequence_manifest_v2_multipos.csv'
BASE_SEQUENCE_MANIFEST_PATH = DATA_ROOT / 'sequence_manifest_v2.csv'
BASE_FRAME_INDEX_PATH = DATA_ROOT / 'frame_cache_index_v2.csv'

TRAIN_CACHE_ROOT = DATA_ROOT / 'processed_v2' / 'frames16_wide_224x320_v2_multipos_train'
FRAME_INDEX_PATH = DATA_ROOT / 'frame_cache_index_v2_multipos.csv'
SEQUENCE_STATUS_PATH = DATA_ROOT / 'sequence_cache_status_v2_multipos.csv'
SUMMARY_PATH = DATA_ROOT / 'frame_cache_summary_v2_multipos.json'
PREVIEW_PATH = DATA_ROOT / 'frame_preview_v2_multipos.jpg'

TARGET_HEIGHT = 224
TARGET_WIDTH = 320
NUM_FRAMES = 16
JPEG_QUALITY = 95
PREPROCESSING_VERSION = 'v2_multipos_rgb_letterbox_replicate_224x320'

assert MULTIPOS_MANIFEST_PATH.exists(), 'Run notebook 18 first.'
assert BASE_SEQUENCE_MANIFEST_PATH.exists(), 'The frozen V2 manifest is missing.'
assert BASE_FRAME_INDEX_PATH.exists(), 'Run notebook 09 first; validation cache is required.'
TRAIN_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
print(f'New train-frame cache: {TRAIN_CACHE_ROOT}')

New train-frame cache: P:\NexarCollisionData\processed_v2\frames16_wide_224x320_v2_multipos_train


c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
sequence_manifest = pd.read_csv(MULTIPOS_MANIFEST_PATH).copy()
base_sequences = pd.read_csv(BASE_SEQUENCE_MANIFEST_PATH).copy()
base_frame_index = pd.read_csv(BASE_FRAME_INDEX_PATH).copy()
timestamp_columns = [f'timestamp_{index:02d}' for index in range(NUM_FRAMES)]

required_columns = {
    'sequence_id', 'video_id', 'video_path', 'label', 'split', 'duration',
    'window_start', 'window_end', 'num_frames', 'sequence_variant', *timestamp_columns,
}
assert required_columns.issubset(sequence_manifest.columns), sorted(required_columns - set(sequence_manifest.columns))

for table in (sequence_manifest, base_sequences, base_frame_index):
    table['video_id'] = table['video_id'].astype(str)
    table['label'] = table['label'].astype(int)
sequence_manifest['num_frames'] = sequence_manifest['num_frames'].astype(int)
base_frame_index['frame_index'] = base_frame_index['frame_index'].astype(int)
base_frame_index['frame_valid'] = base_frame_index['frame_valid'].astype(str).str.lower().eq('true')

assert len(sequence_manifest) == 1560
assert sequence_manifest['sequence_id'].is_unique
assert sequence_manifest['num_frames'].eq(NUM_FRAMES).all()
assert sequence_manifest.groupby(['split', 'label']).size().to_dict() == {
    ('train', 0): 720, ('train', 1): 720, ('validation', 0): 60, ('validation', 1): 60,
}

train_sequences = sequence_manifest.loc[sequence_manifest['split'].eq('train')].copy()
validation_sequences = sequence_manifest.loc[sequence_manifest['split'].eq('validation')].copy()
base_validation_frames = base_frame_index.loc[base_frame_index['split'].eq('validation')].copy()
assert len(train_sequences) == 1440 and len(validation_sequences) == 120
assert len(base_validation_frames) == 120 * NUM_FRAMES
assert base_validation_frames['frame_valid'].all()
assert base_validation_frames['frame_path'].map(lambda value: Path(value).is_file()).all()

base_validation_by_video = {
    video_id: group.sort_values('frame_index').reset_index(drop=True)
    for video_id, group in base_validation_frames.groupby('video_id', sort=False)
}
assert set(validation_sequences['video_id']) == set(base_validation_by_video)
print('Sequences in the multi-position cache:')
display(pd.crosstab(sequence_manifest['split'], sequence_manifest['label']))

Sequences in the multi-position cache:


label,0,1
split,,
train,720,720
validation,60,60


In [3]:
def class_name(label: int) -> str:
    return 'positive' if int(label) == 1 else 'negative'

def variant_key(row: pd.Series) -> str:
    suffix = str(row.sequence_id).rsplit('_', 1)[-1]
    assert suffix in {'p0', 'p1', 'p2', 'n0', 'n1', 'n2'}
    return suffix

def output_dir_for_train_sequence(row: pd.Series) -> Path:
    return TRAIN_CACHE_ROOT / 'train' / class_name(row.label) / f'{int(row.video_id):05d}' / variant_key(row)

def output_path_for_train_frame(row: pd.Series, frame_index: int) -> Path:
    return output_dir_for_train_sequence(row) / f'frame_{frame_index:02d}.jpg'

def resize_letterbox_rgb(frame_rgb: np.ndarray) -> np.ndarray:
    height, width = frame_rgb.shape[:2]
    scale = min(TARGET_WIDTH / width, TARGET_HEIGHT / height)
    resized_width = max(1, int(round(width * scale)))
    resized_height = max(1, int(round(height * scale)))
    interpolation = cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR
    resized = cv2.resize(frame_rgb, (resized_width, resized_height), interpolation=interpolation)
    pad_x = TARGET_WIDTH - resized_width
    pad_y = TARGET_HEIGHT - resized_height
    left, right = pad_x // 2, pad_x - (pad_x // 2)
    top, bottom = pad_y // 2, pad_y - (pad_y // 2)
    return cv2.copyMakeBorder(resized, top, bottom, left, right, cv2.BORDER_REPLICATE)

def decode_rgb_at_timestamp(cap: cv2.VideoCapture, timestamp: float, fps: float) -> tuple[np.ndarray | None, str, float]:
    step = 1.0 / fps if np.isfinite(fps) and fps > 0 else 1.0 / 30.0
    for attempt, offset in enumerate((0.0, step, -step, 2 * step)):
        resolved_timestamp = max(0.0, timestamp + offset)
        cap.set(cv2.CAP_PROP_POS_MSEC, resolved_timestamp * 1000.0)
        ok, frame_bgr = cap.read()
        if ok and frame_bgr is not None:
            status = 'exact' if attempt == 0 else f'seek_fallback_{attempt}'
            return cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB), status, resolved_timestamp
    return None, 'decode_failed', np.nan

def complete_existing_train_cache(row: pd.Series) -> bool:
    for frame_index in range(NUM_FRAMES):
        output_path = output_path_for_train_frame(row, frame_index)
        cached_bgr = cv2.imread(str(output_path), cv2.IMREAD_COLOR) if output_path.is_file() else None
        if cached_bgr is None or cached_bgr.shape[:2] != (TARGET_HEIGHT, TARGET_WIDTH):
            return False
    return True

In [4]:
frame_records = []
sequence_status_records = []

for _, row in tqdm(sequence_manifest.iterrows(), total=len(sequence_manifest), desc='Caching multi-position frames'):
    timestamps = [float(row[column]) for column in timestamp_columns]

    # Validation timestamps are deliberately identical to V2-W2. Reuse the checked cache, never re-decode it.
    if row.split == 'validation':
        source_frames = base_validation_by_video[row.video_id]
        assert len(source_frames) == NUM_FRAMES
        assert source_frames['label'].eq(int(row.label)).all()
        assert np.allclose(source_frames['requested_timestamp'].to_numpy(float), np.asarray(timestamps))
        for frame_index, source_frame in source_frames.iterrows():
            frame_records.append({
                'sequence_id': row.sequence_id, 'video_id': row.video_id, 'video_path': row.video_path,
                'label': row.label, 'split': row.split, 'sequence_variant': row.sequence_variant,
                'frame_index': int(frame_index), 'requested_timestamp': timestamps[frame_index],
                'resolved_timestamp': float(source_frame.resolved_timestamp), 'frame_path': source_frame.frame_path,
                'frame_valid': True, 'decode_status': 'reused_v2_validation_cache',
                'frame_source': 'v2_w2_validation_cache', 'width': TARGET_WIDTH, 'height': TARGET_HEIGHT,
                'preprocessing_version': PREPROCESSING_VERSION,
            })
        sequence_status_records.append({
            'sequence_id': row.sequence_id, 'video_id': row.video_id, 'label': row.label, 'split': row.split,
            'sequence_variant': row.sequence_variant, 'cache_status': 'reused_v2_validation_cache',
            'valid_frames': NUM_FRAMES, 'invalid_frames': 0, 'error_reason': None,
        })
        continue

    output_dir = output_dir_for_train_sequence(row)
    output_dir.mkdir(parents=True, exist_ok=True)
    if complete_existing_train_cache(row):
        for frame_index, timestamp in enumerate(timestamps):
            frame_records.append({
                'sequence_id': row.sequence_id, 'video_id': row.video_id, 'video_path': row.video_path,
                'label': row.label, 'split': row.split, 'sequence_variant': row.sequence_variant,
                'frame_index': frame_index, 'requested_timestamp': timestamp, 'resolved_timestamp': timestamp,
                'frame_path': str(output_path_for_train_frame(row, frame_index)), 'frame_valid': True,
                'decode_status': 'existing_complete_train_cache', 'frame_source': 'existing_multipos_train_cache',
                'width': TARGET_WIDTH, 'height': TARGET_HEIGHT, 'preprocessing_version': PREPROCESSING_VERSION,
            })
        sequence_status_records.append({
            'sequence_id': row.sequence_id, 'video_id': row.video_id, 'label': row.label, 'split': row.split,
            'sequence_variant': row.sequence_variant, 'cache_status': 'skipped_complete_train_cache',
            'valid_frames': NUM_FRAMES, 'invalid_frames': 0, 'error_reason': None,
        })
        continue

    cap = cv2.VideoCapture(str(row.video_path))
    fps = float(cap.get(cv2.CAP_PROP_FPS)) if cap.isOpened() else np.nan
    previous_rgb = None
    valid_frames, invalid_frames, errors = 0, 0, []
    if not cap.isOpened():
        errors.append('cannot_open')

    for frame_index, timestamp in enumerate(timestamps):
        output_path = output_path_for_train_frame(row, frame_index)
        frame_rgb, decode_status, resolved_timestamp = None, 'cannot_open', np.nan
        if cap.isOpened():
            frame_rgb, decode_status, resolved_timestamp = decode_rgb_at_timestamp(cap, timestamp, fps)
        frame_valid = frame_rgb is not None
        if frame_rgb is None and previous_rgb is not None:
            frame_rgb = previous_rgb.copy()
            decode_status = 'repeated_previous_after_decode_failure'
        elif frame_rgb is None:
            errors.append(f'frame_{frame_index:02d}:{decode_status}')

        if frame_rgb is not None:
            processed_rgb = resize_letterbox_rgb(frame_rgb)
            write_ok = cv2.imwrite(str(output_path), cv2.cvtColor(processed_rgb, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
            if write_ok:
                frame_path = str(output_path)
                previous_rgb = frame_rgb
            else:
                errors.append(f'frame_{frame_index:02d}:write_failed')
                frame_valid, frame_path = False, None
        else:
            frame_path = None

        valid_frames += int(frame_valid)
        invalid_frames += int(not frame_valid)
        frame_records.append({
            'sequence_id': row.sequence_id, 'video_id': row.video_id, 'video_path': row.video_path,
            'label': row.label, 'split': row.split, 'sequence_variant': row.sequence_variant,
            'frame_index': frame_index, 'requested_timestamp': timestamp, 'resolved_timestamp': resolved_timestamp,
            'frame_path': frame_path, 'frame_valid': frame_valid, 'decode_status': decode_status,
            'frame_source': 'new_multipos_train_decode', 'width': TARGET_WIDTH, 'height': TARGET_HEIGHT,
            'preprocessing_version': PREPROCESSING_VERSION,
        })
    cap.release()
    sequence_status_records.append({
        'sequence_id': row.sequence_id, 'video_id': row.video_id, 'label': row.label, 'split': row.split,
        'sequence_variant': row.sequence_variant, 'cache_status': 'processed_train_sequence',
        'valid_frames': valid_frames, 'invalid_frames': invalid_frames,
        'error_reason': None if not errors else ';'.join(errors),
    })

frame_cache_index = pd.DataFrame(frame_records).sort_values(['split', 'sequence_id', 'frame_index']).reset_index(drop=True)
sequence_cache_status = pd.DataFrame(sequence_status_records).sort_values(['split', 'sequence_id']).reset_index(drop=True)
frame_cache_index.to_csv(FRAME_INDEX_PATH, index=False)
sequence_cache_status.to_csv(SEQUENCE_STATUS_PATH, index=False)
print(f'Frame index: {FRAME_INDEX_PATH}')
print(f'Sequence status: {SEQUENCE_STATUS_PATH}')

Caching multi-position frames: 100%|██████████| 1560/1560 [4:11:09<00:00,  9.66s/it]     


Frame index: P:\NexarCollisionData\frame_cache_index_v2_multipos.csv
Sequence status: P:\NexarCollisionData\sequence_cache_status_v2_multipos.csv


In [5]:
assert len(frame_cache_index) == len(sequence_manifest) * NUM_FRAMES
assert len(sequence_cache_status) == len(sequence_manifest)
assert frame_cache_index.groupby('sequence_id').size().eq(NUM_FRAMES).all()
assert frame_cache_index['sequence_id'].nunique() == len(sequence_manifest)

valid_frame_count = int(frame_cache_index['frame_valid'].sum())
invalid_frame_count = int((~frame_cache_index['frame_valid']).sum())
invalid_sequence_count = int((sequence_cache_status['invalid_frames'] > 0).sum())
shape_errors = 0
for frame_path in frame_cache_index.loc[frame_cache_index['frame_valid'], 'frame_path']:
    image_bgr = cv2.imread(str(frame_path), cv2.IMREAD_COLOR)
    if image_bgr is None or image_bgr.shape[:2] != (TARGET_HEIGHT, TARGET_WIDTH):
        shape_errors += 1

summary = {
    'source_sequence_manifest': str(MULTIPOS_MANIFEST_PATH),
    'train_cache_root': str(TRAIN_CACHE_ROOT),
    'sequences_total': int(len(sequence_manifest)),
    'sequences_new_train_decode': int(len(train_sequences)),
    'sequences_validation_reused': int(len(validation_sequences)),
    'frames_expected': int(len(sequence_manifest) * NUM_FRAMES),
    'frames_new_train_decode': int(len(train_sequences) * NUM_FRAMES),
    'frames_validation_reused': int(len(validation_sequences) * NUM_FRAMES),
    'frames_valid': valid_frame_count,
    'frames_invalid': invalid_frame_count,
    'sequences_with_invalid_frames': invalid_sequence_count,
    'image_shape_errors': int(shape_errors),
    'target_height': TARGET_HEIGHT,
    'target_width': TARGET_WIDTH,
    'resize_mode': 'letterbox_with_replicated_edge_padding',
    'color_order_before_model_normalization': 'RGB',
    'preprocessing_version': PREPROCESSING_VERSION,
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print(f'Summary: {SUMMARY_PATH}')
display(pd.DataFrame([summary]))
display(sequence_cache_status.groupby(['split', 'label', 'cache_status'])[['valid_frames', 'invalid_frames']].sum())
display(frame_cache_index.groupby(['split', 'frame_source', 'decode_status']).size().rename('frames').to_frame())

assert invalid_frame_count == 0, 'Inspect frame_cache_index_v2_multipos.csv before training.'
assert shape_errors == 0, 'Some cached images have an unexpected shape.'

Summary: P:\NexarCollisionData\frame_cache_summary_v2_multipos.json


,source_sequence_manifest,train_cache_root,sequences_total,sequences_new_train_decode,sequences_validation_reused,frames_expected,frames_new_train_decode,frames_validation_reused,frames_valid,frames_invalid,sequences_with_invalid_frames,image_shape_errors,target_height,target_width,resize_mode,color_order_before_model_normalization,preprocessing_version
0,P:\NexarCollisionData\sequence_manifest_v2_mul...,P:\NexarCollisionData\processed_v2\frames16_wi...,1560,1440,120,24960,23040,1920,24960,0,0,0,224,320,letterbox_with_replicated_edge_padding,RGB,v2_multipos_rgb_letterbox_replicate_224x320


valid_frames  invalid_frames
split      label cache_status                                            
train      0     processed_train_sequence           11520               0
           1     processed_train_sequence           11520               0
validation 0     reused_v2_validation_cache           960               0
           1     reused_v2_validation_cache           960               0

,,,frames
split,frame_source,decode_status,
train,new_multipos_train_decode,exact,23040
validation,v2_w2_validation_cache,reused_v2_validation_cache,1920


In [6]:
preview_sequences = sequence_manifest.loc[
    sequence_manifest['split'].eq('train') & sequence_manifest['label'].eq(1)
].sort_values(['sequence_variant', 'video_id'], key=lambda values: values.astype(int) if values.name == 'video_id' else values)
preview_sequences = preview_sequences.groupby('sequence_variant', sort=True).head(1)
preview_images = []
for _, sequence_row in preview_sequences.iterrows():
    frame_row = frame_cache_index.loc[
        frame_cache_index['sequence_id'].eq(sequence_row.sequence_id) & frame_cache_index['frame_index'].eq(8)
    ].iloc[0]
    image_bgr = cv2.imread(str(frame_row.frame_path), cv2.IMREAD_COLOR)
    cv2.putText(image_bgr, f'{sequence_row.sequence_variant} | t={frame_row.requested_timestamp:.2f}s', (8, 22),
                cv2.FONT_HERSHEY_SIMPLEX, 0.48, (255, 255, 255), 1, cv2.LINE_AA)
    preview_images.append(image_bgr)

preview_bgr = cv2.hconcat(preview_images)
cv2.imwrite(str(PREVIEW_PATH), preview_bgr, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
print(f'Preview: {PREVIEW_PATH}')
display(PREVIEW_PATH)

Preview: P:\NexarCollisionData\frame_preview_v2_multipos.jpg


WindowsPath('P:/NexarCollisionData/frame_preview_v2_multipos.jpg')